In [1]:
"""
====================================================================
DOWNSTREAM CLASSIFICATION + FOUNDATION MODELS + EXPLAINABLE AI
Cross-Centre Generalization on Camelyon17-WILDS
====================================================================

EXTENDS the original downstream-classification pipeline (Phase 0-5) with
the two things the project vision (Sections 17, 19, 26-29 of the project
writeup) calls for and the original notebook didn't have yet:

  (A) THREE HISTOPATHOLOGY FOUNDATION MODELS, frozen, as extra columns in
      every comparison table (primary eval + leave-one-centre-out):
        - Kaiko ViT  (1aurent/..kaiko_ai_fms)  -- ViT-S/16 (DINO), OPEN
        - Phikon     (owkin/phikon)           -- ViT-B/16 (iBOT), OPEN
        - Lunit ViT  (1aurent/..lunit_dino)   -- ViT-S/16 (DINO), OPEN
      These join the existing recon_resnet18 / recon_vit_small /
      recon_swin_tiny / baseline_imagenet / baseline_random models in the
      SAME feature-extraction -> linear-probe -> LOCO pipeline, so every
      table and figure now has 8 columns instead of 5. All three foundation
      models here are OPEN weights -- no HF_TOKEN / gated-licence step needed
      (UNI was swapped for phikon-v2 after UNI's gated repo returned a 401;
      phikon-v2 was then swapped for kaiko_vits16, a ViT-Small/16 pathology
      model, after phikon-v2's ViT-L/16 size made Phase 1 take ~5.3h alone).

  (B) EXPLAINABLE-AI PHASE (Phase 6), exactly as scoped in the project
      writeup Section 26-29:
        - Grad-CAM for every CNN-style / spatial-feature-map model
          (recon_resnet18, baseline_imagenet, recon_swin_tiny) -- computed
          w.r.t. the SAME trained linear-probe tumour logit used for the
          classification numbers above (the StandardScaler is folded into
          the backbone's raw pooled feature so the CAM is faithful to the
          actual decision, not a re-trained proxy head).
        - Attention-rollout for every Transformer-with-global-attention
          model (recon_vit_small, kaiko_vits16, phikon, lunit_vits16) -- the
          "appropriate attention visualization" the writeup calls for as
          the Transformer counterpart to Grad-CAM.
        - A QUANTITATIVE proxy for the IoU/Dice-vs-tumour-mask idea in
          Section 28: Camelyon17-WILDS's patch label is defined by
          whether the CENTER 32x32 px region of each 96x96 patch is
          tumour (not by a full per-pixel mask, which these pre-tiled
          patches don't ship with). So instead of an unavailable mask, we
          measure what fraction of each explanation's attribution mass
          falls inside that same center region -- "center-mass ratio". A
          model attending to the diagnostically-relevant region should
          score close to the region's area fraction (~11%) or higher; one
          drifting onto stain/background artefacts scores lower. Computed
          separately for in-domain (train-centre) and held-out-centre
          correctly-classified tumour patches, per model -- directly
          testing Section 29's question ("does the reconstruction model
          stay aligned with the tumour region when the hospital changes,
          while the baseline drifts?").
        - Side-by-side montage figures (in-domain row vs held-out-centre
          row) per model, plus a summary bar chart of center-mass ratio
          by model x split.

Also added -- the four items needed to make the core "does reconstruction beat
baselines on held-out centres" experiment actually defensible in a paper:

  (C) PATIENT-LEVEL BOOTSTRAP 95% CIs on every AUROC (primary, LOCO, and the
      stain-aug baseline). Patches from the same patient/slide aren't independent
      samples, so resampling patches directly overstates confidence -- this
      resamples PATIENTS with replacement instead (keeping each sampled patient's
      patches together), the statistically valid version for this dataset.

  (D) STAIN-AUGMENTATION BASELINE (`baseline_stain_aug`), trained END-TO-END with
      HED-space stain augmentation directly for tumour classification on each LOCO
      fold's train centres -- the one mandatory comparison that genuinely needed
      new training code, not just another frozen feature extractor.

  (E) MULTI-SEED LEAVE-ONE-CENTRE-OUT: every frozen-feature model's LOCO fold is
      re-fit across Config.SEEDS (cheap -- probe refit only), giving mean +/- std
      AUROC per model per held-out centre instead of one single-seed number.

Everything from the original notebook (Phase 0 checkpoint check, Phase A
diagnostics, Phase 1 feature extraction+caching, Phase 2 primary eval,
Phase 3 leave-one-centre-out, Phase 4/5 figures+CSV) is kept as-is; this
file is a superset, not a replacement.

Kaggle notebook -- paste and run. Needs `timm`, `transformers`,
`huggingface_hub` (all normally preinstalled on Kaggle images; if not:
`pip install timm transformers huggingface_hub`).
====================================================================
"""

import os
import pickle
import gc
import random
import warnings
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    precision_score, recall_score, balanced_accuracy_score,
    confusion_matrix, matthews_corrcoef,
)
from sklearn.utils import shuffle as sk_shuffle
from tqdm.auto import tqdm
from PIL import Image as PILImage
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

warnings.filterwarnings('ignore')


# ====================================================================
# CONFIG
# ====================================================================
class Config:
    HF_DATASET = "wltjr1007/Camelyon17-WILDS"

    CHECKPOINT_DIR = "/kaggle/input/datasets/peesarisathvikreddy/histopathology-reconstruction-checkpoints"
    FEATURE_DIR = "/kaggle/working/features"
    RESULTS_DIR = "/kaggle/working/downstream_results"
    XAI_DIR = "/kaggle/working/downstream_results/xai"

    PRETRAINED_MODELS = {
        'recon_resnet18': {
            'checkpoint': f"{CHECKPOINT_DIR}/v3_resnet18_best.pt",
            'backbone': 'resnet18',
            'input_size': 96,
        },
        'recon_vit_small': {
            'checkpoint': f"{CHECKPOINT_DIR}/v3_vit_small_best.pt",
            'backbone': 'vit_small',
            'input_size': 224,
        },
        'recon_swin_tiny': {
            'checkpoint': f"{CHECKPOINT_DIR}/v2_swin_tiny_best.pt",
            'backbone': 'swin_tiny',
            'input_size': 224,
        },
    }

    # ---- (A) THREE HISTOPATHOLOGY FOUNDATION MODELS, frozen ----
    # 'loader': 'timm' uses timm's hf-hub loader (hub_id must start with "hf-hub:");
    #           'transformers' uses transformers.AutoModel + CLS-token pooling.
    # 'gated' just controls the warning message printed if loading fails.
    FOUNDATION_MODELS = {
        # Swapped out UNI (gated -- needs HF licence acceptance + HF_TOKEN, which
        # failed with a 401) and then phikon-v2 (open, but ViT-L/16 0.3B params --
        # took ~5.3h alone to extract on a T4, blowing the 12h Kaggle budget).
        # kaiko_vits16 is a ViT-Small/16 (22M params) pathology foundation model
        # from Kaiko.ai's "Towards Large-Scale Training of Pathology Foundation
        # Models" work, mirrored open (no login) via the same 1aurent HF account
        # as lunit_vits16 below -- same size class, same ~30min extraction cost.
        'kaiko_vits16': {
            'hub_id': 'hf-hub:1aurent/vit_small_patch16_224.kaiko_ai_towards_large_pathology_fms',
            'loader': 'timm',
            'input_size': 224,
            'gated': False,
        },
        'phikon': {
            'hub_id': 'owkin/phikon',
            'loader': 'transformers',
            'input_size': 224,
            'gated': False,
        },
        'lunit_vits16': {
            'hub_id': 'hf-hub:1aurent/vit_small_patch16_224.lunit_dino',
            'loader': 'timm',
            'input_size': 224,
            'gated': False,
        },
    }

    TRAIN_CENTERS = [0, 1, 3, 4]
    TEST_CENTER = 2
    ALL_CENTERS = [0, 1, 2, 3, 4]

    EXTRACT_BATCH_SIZE = 64
    NUM_WORKERS = 2
    SEED = 42
    SEEDS = [42, 123, 2024]
    LR_MAX_ITER = 2000
    LR_C = 1.0

    # ---- (B) EXPLAINABILITY ----
    # Which models get an explanation map, and which mechanism they use.
    # 'cam'  -> Grad-CAM (needs a spatial feature map + the trained linear-probe weights)
    # 'attn' -> attention-rollout (needs the transformer's per-layer attention weights)
    XAI_MODELS = {
        'recon_resnet18':    {'type': 'cam',  'arch': 'resnet'},
        'baseline_imagenet': {'type': 'cam',  'arch': 'resnet'},
        'recon_swin_tiny':   {'type': 'cam',  'arch': 'swin'},
        'recon_vit_small':   {'type': 'attn', 'arch': 'vit_timm'},
        'kaiko_vits16':      {'type': 'attn', 'arch': 'vit_timm'},
        'lunit_vits16':      {'type': 'attn', 'arch': 'vit_timm'},
        'phikon':            {'type': 'attn', 'arch': 'vit_hf'},
        # 'baseline_stain_aug' deliberately NOT included here: it has its own real
        # classification head (nn.Linear(512,1) trained end-to-end), not a folded
        # frozen-feature + linear-probe pair, so fold_scaler_into_linear() doesn't
        # apply. Grad-CAM against its own head would need a small separate helper
        # (same target layer, but gradient of the sigmoid logit directly rather than
        # a reconstructed logistic-regression decision) -- worth adding once the
        # frozen-model XAI results above are validated.
    }
    XAI_N_SAMPLES = 6          # correctly-classified tumour patches per (model, split)
    # Camelyon17-WILDS's patch label is defined by the CENTER 32x32 px region of the
    # ORIGINAL 96x96 patch -- 32/96 = 1/3 of the width/height, centered. This fraction is
    # what "diagnostically relevant region" means for this dataset, independent of
    # whatever input_size a given backbone resizes to.
    XAI_CENTER_FRAC = 32.0 / 96.0
    XAI_SEED = 123

    # ---- (C) PATIENT/SLIDE-LEVEL BOOTSTRAP CIs ----
    # Patches from the same patient/slide are not independent samples (writeup Section
    # 33), so every AUROC below is also reported with a 95% CI computed by resampling
    # PATIENTS with replacement (keeping each sampled patient's patches together), not
    # by resampling patches independently.
    BOOTSTRAP_N_RESAMPLES = 1000
    BOOTSTRAP_SEED = 7

    # ---- (D) STAIN-AUGMENTATION BASELINE ----
    # A resnet18 trained END-TO-END (not a frozen+probe pipeline like everything else)
    # directly for tumour classification, with heavy stain/colour augmentation, on each
    # LOCO fold's train centres -- the mandatory comparison from the writeup Section 20.
    # This is real training, not feature extraction, so it is the expensive part of this
    # script: kept deliberately small (few epochs, resnet18 only) to stay Kaggle-feasible
    # across 5 LOCO folds. Raise STAIN_AUG_EPOCHS if train loss hasn't flattened.
    STAIN_AUG_EPOCHS = 10
    STAIN_AUG_LR = 1e-4
    STAIN_AUG_BATCH_SIZE = 64
    STAIN_AUG_INPUT_SIZE = 96
    STAIN_AUG_SEED = 42

    # ---- (E) MULTI-SEED LOCO ----
    # Re-fits every frozen model's linear probe (cheap -- no re-extraction, no
    # re-training) across Config.SEEDS for every LOCO fold, so the LOCO table can report
    # mean +/- std across seeds instead of one single-seed number.
    RUN_MULTISEED_LOCO = True


for _d in (Config.FEATURE_DIR, Config.RESULTS_DIR, Config.XAI_DIR):
    os.makedirs(_d, exist_ok=True)


def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def worker_init_fn(worker_id):
    seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(seed); random.seed(seed)


# ====================================================================
# PHASE 0 -- CHECKPOINT VERIFICATION
# ====================================================================
def verify_checkpoints():
    print("=" * 70)
    print("PHASE 0 -- CHECKPOINT VERIFICATION")
    print("=" * 70)
    print(f"Looking in: {Config.CHECKPOINT_DIR}\n")

    alt_paths = [
        Config.CHECKPOINT_DIR,
        "/kaggle/input/histopathology-reconstruction-checkpoints",
        "/kaggle/input/datasets/peesarisathvikreddy/histopathology-reconstruction-checkpoints",
    ]
    found_dir = None
    for p in alt_paths:
        if os.path.exists(p):
            found_dir = p
            break

    if found_dir is None:
        print("X Checkpoint directory not found.")
        return False

    print(f"OK Found: {found_dir}")
    if found_dir != Config.CHECKPOINT_DIR:
        Config.CHECKPOINT_DIR = found_dir
        for name, cfg in Config.PRETRAINED_MODELS.items():
            cfg['checkpoint'] = f"{found_dir}/{os.path.basename(cfg['checkpoint'])}"

    for name, cfg in Config.PRETRAINED_MODELS.items():
        if os.path.exists(cfg['checkpoint']):
            size_mb = os.path.getsize(cfg['checkpoint']) / 1e6
            print(f"  OK {name:20s} ({size_mb:.1f} MB)")
        else:
            print(f"  X  {name:20s} NOT FOUND")
    print()
    return True


# ====================================================================
# DATA LOADING
# ====================================================================
def load_full_dataset():
    from datasets import load_dataset, concatenate_datasets
    ds = load_dataset(Config.HF_DATASET)
    parts = [ds[k] for k in ds.keys()]
    full = concatenate_datasets(parts) if len(parts) > 1 else parts[0]
    print(f"OK Loaded {len(full):,} patches")
    return full


class PatchDataset(Dataset):
    def __init__(self, hf_dataset, indices, input_size):
        self.ds = hf_dataset
        self.indices = indices
        self.transform = transforms.Compose([
            transforms.Resize((input_size, input_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406],
                               [0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        item = self.ds[int(idx)]
        img = item['image']
        if img.mode != 'RGB':
            img = img.convert('RGB')
        return {
            'image': self.transform(img),
            'label': int(item['label']),
            'center': int(item['center']),
            'patient': int(item['patient']),
        }


# ====================================================================
# MODEL BUILDERS
# ====================================================================
def build_recon_model(backbone_name, checkpoint_path,
                     num_positions=9, num_permutations=500,
                     dropout=0.2, context_layers=4, input_size=96):
    import torchvision.models as tvm

    class ReconModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.backbone_name = backbone_name
            self.num_positions = num_positions
            self.input_size = input_size

            if backbone_name == 'resnet18':
                w = tvm.ResNet18_Weights.IMAGENET1K_V1
                net, self.feature_dim = tvm.resnet18(weights=w), 512
                self.backbone = nn.Sequential(*list(net.children())[:-1])
                self.pool = nn.AdaptiveAvgPool2d(1)
            elif backbone_name == 'vit_small':
                import timm
                self.backbone = timm.create_model(
                    'vit_small_patch16_224', pretrained=True,
                    num_classes=0, img_size=input_size)
                self.feature_dim, self.pool = 384, None
            elif backbone_name == 'swin_tiny':
                import timm
                self.backbone = timm.create_model(
                    'swin_tiny_patch4_window7_224',
                    pretrained=True, num_classes=0)
                self.feature_dim, self.pool = 768, None
            else:
                raise ValueError(f"Unknown: {backbone_name}")

            self.proj = nn.Sequential(
                nn.Linear(self.feature_dim, self.feature_dim),
                nn.LayerNorm(self.feature_dim))
            self.context = nn.TransformerEncoder(
                nn.TransformerEncoderLayer(
                    d_model=self.feature_dim, nhead=8,
                    dim_feedforward=min(4 * self.feature_dim, 2048),
                    dropout=dropout, batch_first=True, norm_first=True),
                num_layers=context_layers)
            self.slot_embed = nn.Parameter(
                torch.zeros(1, num_positions, self.feature_dim))
            self.mask_token = nn.Parameter(
                torch.zeros(1, 1, self.feature_dim))
            self.perm_head = nn.Linear(
                self.feature_dim * num_positions, num_permutations)
            self.position_head = nn.Sequential(
                nn.Linear(self.feature_dim, 256), nn.GELU(),
                nn.Linear(256, num_positions))

        @torch.no_grad()
        def encode(self, x):
            f = self.backbone(x)
            if f.dim() == 4:
                f = torch.flatten(self.pool(f), 1)
            elif f.dim() == 3:
                f = f.mean(1)
            return f

    model = ReconModel()
    ck = torch.load(checkpoint_path, map_location='cpu')
    state = ck['model_state_dict']
    if list(state.keys())[0].startswith('module.'):
        state = {k.replace('module.', ''): v for k, v in state.items()}
    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"    epoch={ck.get('epoch', '?')}  "
          f"best_pos={ck.get('best_pos', 0)*100:.2f}%")
    return model


def build_imagenet_model():
    import torchvision.models as tvm

    class ImageNetEncoder(nn.Module):
        def __init__(self):
            super().__init__()
            net = tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1)
            self.feature_dim = 512
            self.backbone = nn.Sequential(*list(net.children())[:-1])
            self.pool = nn.AdaptiveAvgPool2d(1)

        @torch.no_grad()
        def encode(self, x):
            f = self.backbone(x)
            if f.dim() == 4:
                f = torch.flatten(self.pool(f), 1)
            return f

    return ImageNetEncoder()


def build_random_model():
    import torchvision.models as tvm

    class RandomEncoder(nn.Module):
        def __init__(self):
            super().__init__()
            net = tvm.resnet18(weights=None)
            self.feature_dim = 512
            self.backbone = nn.Sequential(*list(net.children())[:-1])
            self.pool = nn.AdaptiveAvgPool2d(1)

        @torch.no_grad()
        def encode(self, x):
            f = self.backbone(x)
            if f.dim() == 4:
                f = torch.flatten(self.pool(f), 1)
            return f

    return RandomEncoder()


def build_foundation_model(name, cfg):
    """
    Loads one of the three frozen histopathology foundation models (see
    Config.FOUNDATION_MODELS) with the same `.encode(x) -> (B, D)` interface as the
    recon/baseline models above, so it drops straight into extract_all_features()
    and every downstream probe/LOCO/figure without special-casing.

    All three configured foundation models (kaiko_vits16, phikon, lunit_vits16)
    are open weights on the Hugging Face Hub -- no login / HF_TOKEN required.
    """
    import timm as _timm

    class TimmViTWrapper(nn.Module):
        def __init__(self, hub_id):
            super().__init__()
            self.backbone = _timm.create_model(hub_id, pretrained=True, num_classes=0)
            self.feature_dim = getattr(self.backbone, 'num_features', None) or 768

        @torch.no_grad()
        def encode(self, x):
            f = self.backbone(x)
            return f if f.dim() == 2 else f.mean(1)

    class HFTransformersViTWrapper(nn.Module):
        def __init__(self, hub_id):
            super().__init__()
            from transformers import AutoModel
            self.backbone = AutoModel.from_pretrained(hub_id)
            self.feature_dim = self.backbone.config.hidden_size

        @torch.no_grad()
        def encode(self, x):
            out = self.backbone(pixel_values=x)
            return out.last_hidden_state[:, 0]   # CLS token

    if cfg['loader'] == 'transformers':
        return HFTransformersViTWrapper(cfg['hub_id'])
    return TimmViTWrapper(cfg['hub_id'])


# ====================================================================
# FEATURE EXTRACTION
# ====================================================================
@torch.no_grad()
def extract_features(model, hf_dataset, indices, input_size, device,
                     batch_size=64, num_workers=2, tag='model'):
    model = model.to(device).eval()
    ds = PatchDataset(hf_dataset, indices, input_size)
    loader = DataLoader(
        ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True,
        worker_init_fn=worker_init_fn)

    features, labels, centers, patients = [], [], [], []
    for batch in tqdm(loader, desc=f"extract {tag}"):
        x = batch['image'].to(device, non_blocking=True)
        f = model.encode(x)
        features.append(f.cpu().numpy())
        labels.extend(batch['label'].numpy())
        centers.extend(batch['center'].numpy())
        patients.extend(batch['patient'].numpy())
        del x, f
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return (np.vstack(features), np.array(labels),
            np.array(centers), np.array(patients))


def extract_all_features(hf_dataset, indices):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\nDevice: {device}")

    results = {}

    # --- Reconstruction models ---
    for name, cfg in Config.PRETRAINED_MODELS.items():
        cache_path = f"{Config.FEATURE_DIR}/{name}.pkl"
        if os.path.exists(cache_path):
            print(f"\n[{name}] loading cached features")
            with open(cache_path, 'rb') as f:
                results[name] = pickle.load(f)
            continue

        if not os.path.exists(cfg['checkpoint']):
            print(f"\n[{name}] X checkpoint missing -- skipping")
            continue

        print(f"\n[{name}] loading model...")
        try:
            model = build_recon_model(
                backbone_name=cfg['backbone'],
                checkpoint_path=cfg['checkpoint'],
                input_size=cfg['input_size'])
            print(f"[{name}] extracting...")
            feats, labels, centers, patients = extract_features(
                model, hf_dataset, indices, cfg['input_size'], device,
                batch_size=Config.EXTRACT_BATCH_SIZE, tag=name)
            data = {'features': feats, 'labels': labels,
                    'centers': centers, 'patients': patients,
                    'feature_dim': feats.shape[1], 'source': 'reconstruction'}
            with open(cache_path, 'wb') as f:
                pickle.dump(data, f)
            results[name] = data
            print(f"[{name}] OK {feats.shape}")
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception as e:
            print(f"[{name}] X failed: {e}")
            continue

    # --- ImageNet baseline ---
    cache_path = f"{Config.FEATURE_DIR}/baseline_imagenet.pkl"
    if os.path.exists(cache_path):
        print(f"\n[baseline_imagenet] loading cache")
        with open(cache_path, 'rb') as f:
            results['baseline_imagenet'] = pickle.load(f)
    else:
        print(f"\n[baseline_imagenet] extracting...")
        try:
            model = build_imagenet_model()
            feats, labels, centers, patients = extract_features(
                model, hf_dataset, indices, 96, device,
                batch_size=Config.EXTRACT_BATCH_SIZE, tag='imagenet')
            data = {'features': feats, 'labels': labels,
                    'centers': centers, 'patients': patients,
                    'feature_dim': feats.shape[1], 'source': 'imagenet'}
            with open(cache_path, 'wb') as f:
                pickle.dump(data, f)
            results['baseline_imagenet'] = data
            print(f"[baseline_imagenet] OK {feats.shape}")
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception as e:
            print(f"[baseline_imagenet] X {e}")

    # --- Random baseline ---
    cache_path = f"{Config.FEATURE_DIR}/baseline_random.pkl"
    if os.path.exists(cache_path):
        print(f"\n[baseline_random] loading cache")
        with open(cache_path, 'rb') as f:
            results['baseline_random'] = pickle.load(f)
    else:
        print(f"\n[baseline_random] extracting...")
        try:
            model = build_random_model()
            feats, labels, centers, patients = extract_features(
                model, hf_dataset, indices, 96, device,
                batch_size=Config.EXTRACT_BATCH_SIZE, tag='random')
            data = {'features': feats, 'labels': labels,
                    'centers': centers, 'patients': patients,
                    'feature_dim': feats.shape[1], 'source': 'random'}
            with open(cache_path, 'wb') as f:
                pickle.dump(data, f)
            results['baseline_random'] = data
            print(f"[baseline_random] OK {feats.shape}")
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception as e:
            print(f"[baseline_random] X {e}")

    # --- (A) THREE FOUNDATION MODELS, frozen ---
    for name, cfg in Config.FOUNDATION_MODELS.items():
        cache_path = f"{Config.FEATURE_DIR}/{name}.pkl"
        if os.path.exists(cache_path):
            print(f"\n[{name}] loading cached features")
            with open(cache_path, 'rb') as f:
                results[name] = pickle.load(f)
            continue

        print(f"\n[{name}] loading foundation model ({cfg['hub_id']})...")
        try:
            model = build_foundation_model(name, cfg)
            print(f"[{name}] extracting...")
            feats, labels, centers, patients = extract_features(
                model, hf_dataset, indices, cfg['input_size'], device,
                batch_size=Config.EXTRACT_BATCH_SIZE, tag=name)
            data = {'features': feats, 'labels': labels,
                    'centers': centers, 'patients': patients,
                    'feature_dim': feats.shape[1], 'source': 'foundation'}
            with open(cache_path, 'wb') as f:
                pickle.dump(data, f)
            results[name] = data
            print(f"[{name}] OK {feats.shape}")
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception as e:
            gate_hint = (" (this model is GATED -- accept the licence on "
                         f"huggingface.co/{cfg['hub_id'].split(':')[-1]} and set HF_TOKEN, "
                         "or run `huggingface-cli login`)") if cfg['gated'] else ""
            print(f"[{name}] X failed: {e}{gate_hint}")
            continue

    return results


# ====================================================================
# PHASE A -- DIAGNOSTICS  (unchanged from the original notebook)
# ====================================================================
def diagnose_random_baseline(data):
    print("\n" + "=" * 70)
    print("PHASE A -- DIAGNOSTIC: RANDOM BASELINE")
    print("=" * 70)

    labels = data['baseline_random']['labels']
    centers = data['baseline_random']['centers']
    train_mask = np.isin(centers, Config.TRAIN_CENTERS)
    test_mask = np.isin(centers, Config.TEST_CENTER)

    y_tr = labels[train_mask]
    y_te = labels[test_mask]
    X_tr = data['baseline_random']['features'][train_mask]
    X_te = data['baseline_random']['features'][test_mask]

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)

    clf = LogisticRegression(max_iter=1000, class_weight='balanced',
                             random_state=42, n_jobs=-1)
    clf.fit(X_tr_s, y_tr)
    auroc_real = roc_auc_score(y_te, clf.predict_proba(X_te_s)[:, 1])
    print(f"\n  [1a] Random CNN + REAL labels:      AUROC = {auroc_real:.4f}")

    y_tr_shuf = sk_shuffle(y_tr, random_state=42)
    clf = LogisticRegression(max_iter=1000, class_weight='balanced',
                             random_state=42, n_jobs=-1)
    clf.fit(X_tr_s, y_tr_shuf)
    auroc_shuf = roc_auc_score(y_te, clf.predict_proba(X_te_s)[:, 1])
    print(f"  [1b] Random CNN + SHUFFLED labels:  AUROC = {auroc_shuf:.4f}")

    X_tr_noise = StandardScaler().fit_transform(
        np.random.randn(len(X_tr), X_tr.shape[1]).astype(np.float32))
    X_te_noise = StandardScaler().fit_transform(
        np.random.randn(len(X_te), X_te.shape[1]).astype(np.float32))
    clf = LogisticRegression(max_iter=1000, class_weight='balanced',
                             random_state=42, n_jobs=-1)
    clf.fit(X_tr_noise, y_tr)
    auroc_noise = roc_auc_score(y_te, clf.predict_proba(X_te_noise)[:, 1])
    print(f"  [1c] Pure Gaussian noise:            AUROC = {auroc_noise:.4f}")

    norms = np.linalg.norm(X_tr, axis=1)
    norm_tumor = norms[y_tr == 1].mean()
    norm_normal = norms[y_tr == 0].mean()
    ratio = norm_tumor / (norm_normal + 1e-8)
    print(f"\n  [1d] Feature norm tumor/normal: {norm_tumor:.2f}/{norm_normal:.2f} "
          f"= {ratio:.3f}")
    print(f"       (ratio near 1.0 = no leak; far from 1.0 = possible leak)")

    print("\n  --- VERDICT ---")
    if auroc_real > 0.60 and auroc_shuf < 0.55 and auroc_noise < 0.55:
        print("  OK Random features carry real signal (expected for random-init ResNet)")
    elif auroc_shuf > 0.55:
        print("  FLAG Shuffled labels still give signal -- potential leak")
    elif auroc_noise > 0.55:
        print("  FLAG Even Gaussian noise gives signal -- pipeline issue")

    return {
        'auroc_real_labels': auroc_real,
        'auroc_shuffled_labels': auroc_shuf,
        'auroc_gaussian_noise': auroc_noise,
        'norm_ratio': ratio,
    }


def diagnose_standardization(data):
    print("\n" + "=" * 70)
    print("PHASE A -- DIAGNOSTIC: STANDARDIZATION LEAK")
    print("=" * 70)

    results = {}
    for name in data.keys():
        features = data[name]['features']
        labels = data[name]['labels']
        centers = data[name]['centers']

        train_mask = np.isin(centers, Config.TRAIN_CENTERS)
        test_mask = np.isin(centers, Config.TEST_CENTER)
        X_tr = features[train_mask]
        y_tr = labels[train_mask]
        X_te = features[test_mask]
        y_te = labels[test_mask]

        scaler = StandardScaler()
        X_tr_a = scaler.fit_transform(X_tr)
        X_te_a = scaler.transform(X_te)
        clf = LogisticRegression(max_iter=1000, class_weight='balanced',
                                 random_state=42, n_jobs=-1)
        clf.fit(X_tr_a, y_tr)
        auroc_a = roc_auc_score(y_te, clf.predict_proba(X_te_a)[:, 1])

        clf = LogisticRegression(max_iter=1000, class_weight='balanced',
                                 random_state=42, n_jobs=-1)
        clf.fit(X_tr, y_tr)
        auroc_b = roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1])

        diff = abs(auroc_a - auroc_b)
        status = "OK" if diff < 0.01 else "FLAG"
        print(f"  {name:25s}  std={auroc_a:.4f}  no_std={auroc_b:.4f}  "
              f"|diff|={diff:.4f}  {status}")

        results[name] = {'with_std': auroc_a, 'no_std': auroc_b, 'diff': diff}

    return results


def test_checkpoint_integrity():
    print("\n" + "=" * 70)
    print("PHASE A -- DIAGNOSTIC: CHECKPOINT INTEGRITY")
    print("=" * 70)

    ckpt_v3 = Config.PRETRAINED_MODELS['recon_resnet18']['checkpoint']
    if not os.path.exists(ckpt_v3):
        print("  v3 checkpoint not found")
        return {}

    ck = torch.load(ckpt_v3, map_location='cpu')
    print(f"  v3_resnet18:")
    print(f"    epoch    : {ck.get('epoch', '?')}")
    print(f"    best_pos : {ck.get('best_pos', 0)*100:.2f}%")
    print(f"    keys     : {len(ck['model_state_dict'])}")

    for k, v in ck['model_state_dict'].items():
        if 'backbone.0.weight' in k:
            print(f"    first conv: shape={tuple(v.shape)}  "
                  f"mean={v.mean():.5f}  std={v.std():.5f}")
            break

    return {'v3_epoch': ck.get('epoch'), 'v3_best_pos': ck.get('best_pos')}


def run_multi_seed(data, seeds=Config.SEEDS):
    print("\n" + "=" * 70)
    print("PHASE A -- DIAGNOSTIC: MULTI-SEED LINEAR PROBE")
    print("=" * 70)

    results = []
    for name, d in data.items():
        features = d['features']
        labels = d['labels']
        centers = d['centers']

        train_mask = np.isin(centers, Config.TRAIN_CENTERS)
        test_mask = np.isin(centers, Config.TEST_CENTER)
        X_tr = features[train_mask]
        y_tr = labels[train_mask]
        X_te = features[test_mask]
        y_te = labels[test_mask]

        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_te_s = scaler.transform(X_te)

        aurocs = []
        for seed in seeds:
            clf = LogisticRegression(
                max_iter=Config.LR_MAX_ITER, C=Config.LR_C,
                class_weight='balanced', random_state=seed, n_jobs=-1)
            clf.fit(X_tr_s, y_tr)
            auroc = roc_auc_score(y_te, clf.predict_proba(X_te_s)[:, 1])
            aurocs.append(auroc)

        mean = np.mean(aurocs)
        std = np.std(aurocs)
        results.append({
            'model': name,
            'auroc_mean': mean, 'auroc_std': std,
            'auroc_min': min(aurocs), 'auroc_max': max(aurocs),
        })
        print(f"  {name:25s}  AUROC = {mean:.4f} +/- {std:.4f}  "
              f"(min={min(aurocs):.4f}, max={max(aurocs):.4f})")

    return results


# ====================================================================
# PHASE 2/3 -- EVALUATION
# ====================================================================
def compute_metrics(y_true, y_pred, y_prob):
    m = {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'auroc': roc_auc_score(y_true, y_prob),
        'f1': f1_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'mcc': matthews_corrcoef(y_true, y_pred),
    }
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    m['sensitivity'] = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    m['specificity'] = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return m


# ---- (C) PATIENT/SLIDE-LEVEL BOOTSTRAP CI ----
def bootstrap_auroc_by_patient(y_true, y_prob, patient_ids,
                              n_boot=Config.BOOTSTRAP_N_RESAMPLES,
                              seed=Config.BOOTSTRAP_SEED):
    """
    Patch-level bootstrap (resampling patches independently) treats every patch as an
    independent sample, which is wrong -- patches from the same patient/slide are
    correlated (writeup Section 33). This resamples PATIENTS with replacement instead:
    each bootstrap draw picks a set of patient IDs (with repeats), pulls ALL of that
    patient's patches (with repeats for repeated patients), and recomputes AUROC on that
    pooled set. The resulting 2.5/97.5 percentile interval is the patient-cluster-aware
    95% CI that should accompany every AUROC number in a paper built on this dataset.
    Falls back to NaNs (with n_valid_boot=0) if the test set has too few patients or
    only one class is ever present after resampling.
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    patient_ids = np.asarray(patient_ids)
    unique_patients = np.unique(patient_ids)
    idx_by_patient = {p: np.where(patient_ids == p)[0] for p in unique_patients}

    rng = np.random.RandomState(seed)
    boot_aurocs = []
    for _ in range(n_boot):
        sampled = rng.choice(unique_patients, size=len(unique_patients), replace=True)
        idx = np.concatenate([idx_by_patient[p] for p in sampled])
        y_b, p_b = y_true[idx], y_prob[idx]
        if len(np.unique(y_b)) < 2:
            continue
        boot_aurocs.append(roc_auc_score(y_b, p_b))

    if not boot_aurocs:
        return {'auroc_boot_mean': float('nan'), 'auroc_ci_lo': float('nan'),
                'auroc_ci_hi': float('nan'), 'n_boot_valid': 0}
    lo, hi = np.percentile(boot_aurocs, [2.5, 97.5])
    return {'auroc_boot_mean': float(np.mean(boot_aurocs)), 'auroc_ci_lo': float(lo),
            'auroc_ci_hi': float(hi), 'n_boot_valid': len(boot_aurocs)}


def evaluate_linear_probe(features, labels, centers, patients, model_name,
                         train_centers, test_center,
                         C=1.0, max_iter=2000, seed=42, return_probe=False):
    train_mask = np.isin(centers, train_centers)
    test_mask = np.isin(centers, test_center)
    X_tr, y_tr = features[train_mask], labels[train_mask]
    X_te, y_te, pat_te = features[test_mask], labels[test_mask], patients[test_mask]

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)

    clf = LogisticRegression(
        max_iter=max_iter, C=C, class_weight='balanced',
        random_state=seed, n_jobs=-1)
    clf.fit(X_tr_s, y_tr)

    y_pred = clf.predict(X_te_s)
    y_prob = clf.predict_proba(X_te_s)[:, 1]

    metrics = compute_metrics(y_te, y_pred, y_prob)
    metrics.update(bootstrap_auroc_by_patient(y_te, y_prob, pat_te))
    metrics['model'] = model_name
    metrics['train_centers'] = str(list(train_centers))
    metrics['test_center'] = int(test_center[0])
    metrics['n_train'] = int(len(X_tr))
    metrics['n_test'] = int(len(X_te))
    if return_probe:
        return metrics, clf, scaler
    return metrics


def evaluate_all(feature_data, train_centers, test_center, tag='primary',
                seed=42, fitted_probes=None):
    """
    fitted_probes: optional dict to fill with {model_name: (clf, scaler)} for every
    model evaluated here -- used later by the explainability phase to Grad-CAM /
    attention-rollout w.r.t. the EXACT trained tumour-probe decision, not a re-fit
    proxy. Only meaningful/filled when tag='primary' (the split the XAI phase reuses).
    """
    results = []
    for name, data in feature_data.items():
        try:
            if fitted_probes is not None:
                m, clf, scaler = evaluate_linear_probe(
                    data['features'], data['labels'], data['centers'], data['patients'],
                    model_name=name, train_centers=train_centers,
                    test_center=[test_center],
                    C=Config.LR_C, max_iter=Config.LR_MAX_ITER, seed=seed,
                    return_probe=True)
                fitted_probes[name] = (clf, scaler)
            else:
                m = evaluate_linear_probe(
                    data['features'], data['labels'], data['centers'], data['patients'],
                    model_name=name, train_centers=train_centers,
                    test_center=[test_center],
                    C=Config.LR_C, max_iter=Config.LR_MAX_ITER, seed=seed)
            m['split_tag'] = tag
            m['seed'] = seed
            results.append(m)
            print(f"  {name:25s}  AUROC={m['auroc']:.4f}  "
                  f"[{m['auroc_ci_lo']:.4f}, {m['auroc_ci_hi']:.4f}] (patient-boot 95% CI)  "
                  f"F1={m['f1']:.4f}  BalAcc={m['balanced_accuracy']:.4f}")
        except Exception as e:
            print(f"  {name:25s}  X {e}")
    return results


def leave_one_centre_out(feature_data, seed=42):
    rows = []
    for held_out in Config.ALL_CENTERS:
        train = [c for c in Config.ALL_CENTERS if c != held_out]
        print(f"\n{'='*70}")
        print(f"HELD OUT: centre {held_out}  |  TRAIN: {train}  |  seed={seed}")
        print(f"{'='*70}")
        results = evaluate_all(feature_data, train, held_out,
                              tag=f'holdout_{held_out}', seed=seed)
        rows.extend(results)
    return rows


# ---- (E) MULTI-SEED LOCO ----
def run_multiseed_loco(feature_data, seeds=Config.SEEDS):
    """
    Re-runs leave_one_centre_out() once per seed in `seeds`. This is CHEAP for every
    frozen-feature model (recon_*, baseline_imagenet, baseline_random, the 3 foundation
    models) -- only the LogisticRegression probe's random_state changes, no
    re-extraction and no re-training. Returns one long DataFrame with a `seed` column;
    the caller aggregates mean/std across seeds per (model, held_out_centre) to get the
    seed-variance number a single-seed AUROC can't give you.
    """
    print("\n" + "=" * 70)
    print(f"MULTI-SEED LEAVE-ONE-CENTRE-OUT  (seeds={seeds})")
    print("=" * 70)
    all_rows = []
    for seed in seeds:
        print(f"\n----- seed {seed} -----")
        all_rows.extend(leave_one_centre_out(feature_data, seed=seed))
    df = pd.DataFrame(all_rows)
    if df.empty:
        return df, pd.DataFrame()
    summary = df.groupby(['model', 'test_center']).agg(
        auroc_mean=('auroc', 'mean'), auroc_std=('auroc', 'std'),
        auroc_min=('auroc', 'min'), auroc_max=('auroc', 'max'),
    ).reset_index()
    print("\n" + "-" * 70)
    print("MULTI-SEED LOCO SUMMARY (mean +/- std across seeds, per model x held-out centre)")
    print("-" * 70)
    print(summary.to_string(index=False))
    return df, summary


# ====================================================================
# PHASE D -- STAIN-AUGMENTATION BASELINE
# ====================================================================
# Unlike every other model above (frozen backbone -> extract features once -> fit a
# cheap linear probe), this one is trained END-TO-END, directly for tumour
# classification, on each LOCO fold's train centres, with heavy stain/colour
# augmentation. That's the mandatory "does simple stain augmentation already fix
# cross-centre generalization" comparison from the writeup Section 20 -- if it beats
# your reconstruction models, that's still a publishable, honestly-reported result
# (Section 45); if it doesn't, that's your headline finding.
def hed_stain_augment(img_pil, alpha_range=(0.9, 1.1), beta_range=(-0.08, 0.08)):
    """
    Tellez et al. (2018) stain augmentation: perturb the image in HED (Haematoxylin-
    Eosin-DAB) colour space rather than plain RGB, since that's the axis along which
    real inter-hospital staining variation actually happens. Needs scikit-image
    (preinstalled on Kaggle images; `pip install scikit-image` otherwise).
    """
    from skimage.color import rgb2hed, hed2rgb
    arr = np.asarray(img_pil).astype(np.float32) / 255.0
    hed = rgb2hed(arr)
    for c in range(3):
        alpha = np.random.uniform(*alpha_range)
        beta = np.random.uniform(*beta_range)
        hed[..., c] = hed[..., c] * alpha + beta
    rgb = np.clip(hed2rgb(hed), 0, 1)
    return PILImage.fromarray((rgb * 255).astype(np.uint8))


class StainAugDataset(Dataset):
    """Same interface as PatchDataset, but with heavy stain/colour augmentation when
    `train=True` and a plain deterministic resize+normalize when `train=False` (for
    evaluation -- augmenting the test set would make the eval nondeterministic)."""
    def __init__(self, hf_dataset, indices, input_size, train=True):
        self.ds = hf_dataset
        self.indices = indices
        steps = []
        if train:
            steps += [transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
                      transforms.Lambda(lambda im: hed_stain_augment(im)),
                      transforms.ColorJitter(brightness=0.2, contrast=0.2,
                                            saturation=0.2, hue=0.02)]
        steps += [transforms.Resize((input_size, input_size)), transforms.ToTensor(),
                  transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]
        self.transform = transforms.Compose(steps)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        item = self.ds[int(idx)]
        img = item['image']
        if img.mode != 'RGB':
            img = img.convert('RGB')
        return {'image': self.transform(img), 'label': int(item['label']),
                'center': int(item['center']), 'patient': int(item['patient'])}


def build_stain_aug_classifier():
    """ImageNet-initialized resnet18 with a fresh binary head -- trained end-to-end,
    unlike the frozen-backbone-plus-linear-probe pattern everywhere else in this file."""
    import torchvision.models as tvm
    net = tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1)
    net.fc = nn.Linear(net.fc.in_features, 1)
    return net


def train_stain_aug_model(hf_dataset, train_idx, device,
                         epochs=Config.STAIN_AUG_EPOCHS, lr=Config.STAIN_AUG_LR,
                         batch_size=Config.STAIN_AUG_BATCH_SIZE, seed=Config.STAIN_AUG_SEED,
                         tag='fold'):
    set_seed(seed)
    model = build_stain_aug_classifier().to(device)
    ds = StainAugDataset(hf_dataset, train_idx, Config.STAIN_AUG_INPUT_SIZE, train=True)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True,
                        num_workers=Config.NUM_WORKERS, pin_memory=True,
                        worker_init_fn=worker_init_fn)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, epochs * len(loader)))
    criterion = nn.BCEWithLogitsLoss()

    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        for batch in tqdm(loader, desc=f"stain-aug train {tag} ep{epoch + 1}/{epochs}"):
            x = batch['image'].to(device, non_blocking=True)
            y = batch['label'].float().to(device, non_blocking=True)
            opt.zero_grad()
            logit = model(x).squeeze(1)
            loss = criterion(logit, y)
            loss.backward()
            opt.step()
            sched.step()
            total_loss += loss.item() * x.size(0)
        print(f"  [stain-aug {tag}] epoch {epoch + 1}/{epochs}  loss={total_loss / len(ds):.4f}")

    model.eval()
    return model


@torch.no_grad()
def evaluate_stain_aug_model(model, hf_dataset, test_idx, patients_all, device,
                            batch_size=64):
    ds = PatchDataset(hf_dataset, test_idx, Config.STAIN_AUG_INPUT_SIZE)   # deterministic eval transform
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        num_workers=Config.NUM_WORKERS, pin_memory=True,
                        worker_init_fn=worker_init_fn)
    probs, labels = [], []
    for batch in tqdm(loader, desc="stain-aug eval"):
        x = batch['image'].to(device, non_blocking=True)
        logit = model(x).squeeze(1)
        probs.append(torch.sigmoid(logit).cpu().numpy())
        labels.append(batch['label'].numpy())
    probs = np.concatenate(probs)
    y_te = np.concatenate(labels)
    pat_te = patients_all[test_idx]
    preds = (probs >= 0.5).astype(int)

    m = compute_metrics(y_te, preds, probs)
    m.update(bootstrap_auroc_by_patient(y_te, probs, pat_te))
    return m


def run_stain_augmentation_loco(hf_dataset, centers_all, patients_all, device):
    """
    Trains + evaluates the stain-aug baseline on all 5 LOCO folds (retraining from
    scratch each fold, since the train-centre set changes per fold -- 5 full training
    runs total). The fold where held_out == Config.TEST_CENTER is additionally tagged
    'primary' so it lines up with the frozen-model primary-evaluation table.
    """
    print("\n" + "=" * 70)
    print("PHASE D -- STAIN-AUGMENTATION BASELINE (trained end-to-end, 5 LOCO folds)")
    print("=" * 70)
    all_idx = np.arange(len(centers_all))
    rows = []
    for held_out in Config.ALL_CENTERS:
        train_centers = [c for c in Config.ALL_CENTERS if c != held_out]
        train_idx = all_idx[np.isin(centers_all, train_centers)]
        test_idx = all_idx[centers_all == held_out]
        print(f"\n{'-'*70}\n[stain-aug] HELD OUT centre {held_out}  |  TRAIN {train_centers}\n{'-'*70}")

        model = train_stain_aug_model(hf_dataset, train_idx, device, tag=f"holdout{held_out}")
        m = evaluate_stain_aug_model(model, hf_dataset, test_idx, patients_all, device)
        m.update({'model': 'baseline_stain_aug', 'train_centers': str(train_centers),
                 'test_center': int(held_out), 'split_tag': f'holdout_{held_out}',
                 'n_train': int(len(train_idx)), 'n_test': int(len(test_idx)),
                 'seed': Config.STAIN_AUG_SEED})
        rows.append(m)
        print(f"  [stain-aug] held-out {held_out}  AUROC={m['auroc']:.4f}  "
              f"[{m['auroc_ci_lo']:.4f}, {m['auroc_ci_hi']:.4f}]  BalAcc={m['balanced_accuracy']:.4f}")

        if held_out == Config.TEST_CENTER:
            primary_row = dict(m)
            primary_row['split_tag'] = 'primary'
            rows.append(primary_row)

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return rows


# ====================================================================
# FIGURES  (unchanged, now naturally covers 8 models instead of 5)
# ====================================================================
def make_figures(df, prefix):
    prim = df[df['split_tag'] == 'primary'].sort_values('auroc', ascending=False)
    if not prim.empty:
        fig, ax = plt.subplots(figsize=(12, 5))
        colors = ['#2ecc71' if 'recon' in m else
                  '#9b59b6' if m in Config.FOUNDATION_MODELS else
                  '#f39c12' if m == 'baseline_stain_aug' else
                  '#e74c3c' if 'random' in m else '#3498db'
                  for m in prim['model']]
        bars = ax.bar(prim['model'], prim['auroc'], color=colors, edgecolor='black')
        if 'auroc_ci_lo' in prim.columns:
            yerr = np.abs(np.vstack([prim['auroc'] - prim['auroc_ci_lo'],
                                     prim['auroc_ci_hi'] - prim['auroc']]))
            ax.errorbar(prim['model'], prim['auroc'], yerr=yerr, fmt='none',
                       ecolor='black', capsize=4)
        for b, v in zip(bars, prim['auroc']):
            ax.text(b.get_x() + b.get_width()/2, v + 0.005,
                    f'{v:.3f}', ha='center', fontsize=10, fontweight='bold')
        ax.set_ylabel('AUROC')
        ax.set_title('Cross-Centre AUROC (Center 2 held out) -- recon / baseline / foundation models')
        ax.set_ylim(0.5, 1.0)
        ax.grid(axis='y', alpha=0.3)
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.savefig(f"{prefix}_auroc_bar.png", dpi=150)
        plt.close()
        print(f"OK {prefix}_auroc_bar.png")

    lco = df[df['split_tag'].str.startswith('holdout_', na=False)]
    if not lco.empty:
        pivot = lco.pivot_table(index='model', columns='test_center',
                                values='auroc', aggfunc='mean')
        fig, ax = plt.subplots(figsize=(9, 6))
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn',
                    vmin=0.5, vmax=1.0, ax=ax)
        ax.set_title('AUROC by Model x Held-Out Centre')
        plt.tight_layout()
        plt.savefig(f"{prefix}_heatmap.png", dpi=150)
        plt.close()
        print(f"OK {prefix}_heatmap.png")


# ====================================================================
# PHASE 6 -- EXPLAINABLE AI (Grad-CAM + attention-rollout)
# ====================================================================
def fold_scaler_into_linear(clf, scaler):
    """
    The trained tumour probe is: logit = w . scale(feat) + b, where scale(feat) =
    (feat - scaler.mean_) / scaler.scale_. Grad-CAM needs gradients of that SAME
    logit w.r.t. the model's RAW (unscaled) pooled feature, so we fold the
    StandardScaler affine transform into the logistic-regression weight/bias:
        logit = (w / scaler.scale_) . feat + (b - sum(w * scaler.mean_ / scaler.scale_))
    This makes the CAM faithful to the exact decision boundary used for the
    reported AUROC/F1 numbers, not a separately-fit proxy head.
    """
    w = clf.coef_[0]
    b = float(clf.intercept_[0])
    eff_w = w / scaler.scale_
    eff_b = b - float(np.sum(w * scaler.mean_ / scaler.scale_))
    return torch.tensor(eff_w, dtype=torch.float32), torch.tensor(eff_b, dtype=torch.float32)


def encode_with_grad(model, x):
    """
    Grad-enabled replica of the CNN/Swin-style `.encode()` bodies above (which are
    all @torch.no_grad()). Structurally identical pooling logic, so the trained
    linear-probe weights remain valid for whatever this returns.
    """
    f = model.backbone(x)
    if f.dim() == 4:
        f = torch.flatten(model.pool(f), 1) if hasattr(model, 'pool') and model.pool is not None else torch.flatten(f, 1)
    elif f.dim() == 3:
        f = f.mean(1)
    return f


def _cam_target_layer(model, arch):
    """
    arch='resnet': model.backbone is nn.Sequential(conv1,bn1,relu,maxpool,layer1,
    layer2,layer3,layer4,avgpool) -- index 7 is layer4, the last conv block before
    avgpool (index 8). arch='swin': last Swin stage, model.backbone.layers[-1].
    """
    if arch == 'resnet':
        return model.backbone[7]
    if arch == 'swin':
        return model.backbone.layers[-1]
    raise ValueError(f"no CAM target layer registered for arch={arch}")


def grad_cam(model, x, arch, eff_w, eff_b, device):
    """
    Standard Grad-CAM (Selvaraju et al. 2017), computed w.r.t. the trained tumour
    probe's logit (see fold_scaler_into_linear). Works for both plain-CNN spatial
    maps (B,C,H,W) and Swin's token-grid output ((B,H,W,C) or (B,N,C)) via the
    reshape fallback below -- prints the raw shape once if the fallback path is
    used, so a shape mismatch on your specific timm version is easy to spot.
    Returns a (H,W) numpy heatmap normalized to [0,1].
    """
    target_layer = _cam_target_layer(model, arch)
    activation = {}

    def fwd_hook(module, inp, out):
        out.retain_grad()
        activation['act'] = out

    h = target_layer.register_forward_hook(fwd_hook)
    try:
        model.zero_grad(set_to_none=True)
        feat = encode_with_grad(model, x)                       # (1, D), triggers fwd_hook
        logit = torch.matmul(feat[0], eff_w.to(device)) + eff_b.to(device)
        logit.backward()

        act = activation['act'][0]                              # drop batch dim
        grad = activation['act'].grad[0]

        if act.dim() == 3 and act.shape[0] not in (act.shape[-1],) and act.shape[-1] > act.shape[0]:
            # looks like (H, W, C) -- Swin's channels-last stage output -> (C, H, W)
            act = act.permute(2, 0, 1)
            grad = grad.permute(2, 0, 1)
        elif act.dim() == 2:
            # (N, C) tokens, no explicit H/W -- assume a square grid
            n, c = act.shape
            side = int(round(n ** 0.5))
            act = act.reshape(side, side, c).permute(2, 0, 1)
            grad = grad.reshape(side, side, c).permute(2, 0, 1)
        # else: already (C, H, W) -- resnet's conv output, nothing to do

        weights = grad.mean(dim=(1, 2))                          # (C,)
        cam = F.relu((weights[:, None, None] * act).sum(0))
        cam = cam.detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam
    finally:
        h.remove()


def _vit_attn_rollout_timm(model, x, device):
    """
    Attention-rollout (Abnar & Zuidema 2020) for a standard timm VisionTransformer
    (recon_vit_small, lunit_vits16, kaiko_vits16 -- all expose `.backbone.blocks[i].attn`;
    phikon uses the separate `_vit_attn_rollout_hf` below instead).
    Hooks each block's `attn_drop` module to read its INPUT (the post-softmax,
    pre-dropout attention probabilities), then rolls them up into one CLS-to-patch
    relevance map. Returns a (side, side) numpy heatmap over patch tokens (CLS
    token itself excluded), normalized to [0,1].
    """
    attn_maps = []

    def make_hook():
        def hook(module, inp, out):
            attn_maps.append(inp[0].detach())   # (B, heads, N, N), pre-dropout
        return hook

    handles = [blk.attn.attn_drop.register_forward_hook(make_hook())
               for blk in model.backbone.blocks]
    try:
        with torch.no_grad():
            model.backbone(x)
    finally:
        for hd in handles:
            hd.remove()

    if not attn_maps:
        raise RuntimeError("no attention weights captured -- attn_drop hook path "
                            "didn't match this model's block structure")

    n = attn_maps[0].shape[-1]
    result = torch.eye(n, device=device).unsqueeze(0)
    for A in attn_maps:
        A = A.mean(dim=1)                                    # average heads -> (B,N,N)
        A = A + torch.eye(n, device=device).unsqueeze(0)      # residual connection
        A = A / A.sum(dim=-1, keepdim=True)
        result = torch.bmm(A, result)

    cls_relevance = result[0, 0, 1:]                          # CLS row, drop CLS-self entry
    side = int(round((n - 1) ** 0.5))
    heatmap = cls_relevance[:side * side].reshape(side, side).cpu().numpy()
    heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
    return heatmap


def _vit_attn_rollout_hf(model, x, device):
    """Same idea as above, but for a `transformers` ViTModel (Phikon) which returns
    per-layer attentions directly via output_attentions=True -- no hooking needed."""
    with torch.no_grad():
        out = model.backbone(pixel_values=x, output_attentions=True)
    attn_maps = out.attentions                                # tuple of (B, heads, N, N)
    n = attn_maps[0].shape[-1]
    result = torch.eye(n, device=device).unsqueeze(0)
    for A in attn_maps:
        A = A.mean(dim=1)
        A = A + torch.eye(n, device=device).unsqueeze(0)
        A = A / A.sum(dim=-1, keepdim=True)
        result = torch.bmm(A, result)
    cls_relevance = result[0, 0, 1:]
    side = int(round((n - 1) ** 0.5))
    heatmap = cls_relevance[:side * side].reshape(side, side).cpu().numpy()
    heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
    return heatmap


def attention_rollout(model, x, arch, device):
    if arch == 'vit_hf':
        return _vit_attn_rollout_hf(model, x, device)
    return _vit_attn_rollout_timm(model, x, device)


def explain_one(name, model, x, device, eff_w=None, eff_b=None):
    """Dispatches to Grad-CAM or attention-rollout by Config.XAI_MODELS[name]['type'],
    returns a small (side,side) heatmap in [0,1] -- caller resizes/overlays it."""
    spec = Config.XAI_MODELS[name]
    if spec['type'] == 'cam':
        return grad_cam(model, x, spec['arch'], eff_w, eff_b, device)
    return attention_rollout(model, x, spec['arch'], device)


def center_mass_ratio(heatmap_2d):
    """
    What fraction of this explanation's attribution mass falls inside the CENTER
    32x32-px-equivalent region (Config.XAI_CENTER_FRAC of width/height, centered) --
    the region that actually DEFINES the tumour label in Camelyon17-WILDS. A model
    reading the diagnostically-relevant region should score noticeably above the
    region's own area fraction (~ (1/3)^2 = 11%); one drifting onto stain/background
    artefacts around the edges scores at or below it.
    """
    H, W = heatmap_2d.shape
    ch, cw = max(1, int(round(H * Config.XAI_CENTER_FRAC))), max(1, int(round(W * Config.XAI_CENTER_FRAC)))
    y0, x0 = (H - ch) // 2, (W - cw) // 2
    total = heatmap_2d.sum() + 1e-8
    return float(heatmap_2d[y0:y0 + ch, x0:x0 + cw].sum() / total)


def _load_xai_model(name, device):
    if name in Config.PRETRAINED_MODELS:
        cfg = Config.PRETRAINED_MODELS[name]
        model = build_recon_model(backbone_name=cfg['backbone'],
                                  checkpoint_path=cfg['checkpoint'],
                                  input_size=cfg['input_size'])
        input_size = cfg['input_size']
    elif name == 'baseline_imagenet':
        model = build_imagenet_model()
        input_size = 96
    elif name in Config.FOUNDATION_MODELS:
        cfg = Config.FOUNDATION_MODELS[name]
        model = build_foundation_model(name, cfg)
        input_size = cfg['input_size']
    else:
        raise ValueError(f"no XAI loader for {name}")
    return model.to(device).eval(), input_size


def select_xai_samples(feature_data, name, fitted_probes, n_per_group, seed):
    """Correctly-classified TUMOUR patches, split into in-domain (train centres) and
    held-out-centre (Config.TEST_CENTER) pools, using the EXACT trained primary probe."""
    d = feature_data[name]
    clf, scaler = fitted_probes[name]
    X, y, c = d['features'], d['labels'], d['centers']
    preds = clf.predict(scaler.transform(X))
    correct_tumor = (preds == 1) & (y == 1)

    in_domain_pool = np.where(correct_tumor & np.isin(c, Config.TRAIN_CENTERS))[0]
    held_out_pool = np.where(correct_tumor & (c == Config.TEST_CENTER))[0]

    rng = np.random.RandomState(seed)
    in_sel = rng.choice(in_domain_pool, size=min(n_per_group, len(in_domain_pool)), replace=False) if len(in_domain_pool) else np.array([], dtype=int)
    ho_sel = rng.choice(held_out_pool, size=min(n_per_group, len(held_out_pool)), replace=False) if len(held_out_pool) else np.array([], dtype=int)
    return in_sel, ho_sel


def run_explainability_study(hf_dataset, feature_data, fitted_probes, timestamp):
    """
    PHASE 6 -- for every model in Config.XAI_MODELS: pick a few correctly-classified
    tumour patches from in-domain centres and from the held-out centre, generate a
    Grad-CAM or attention-rollout heatmap for each, overlay it on the patch, save a
    montage (in-domain row vs held-out-centre row), and log the center-mass-ratio
    proxy metric per (model, split). Returns the metric rows as a DataFrame.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print("\n" + "=" * 70)
    print("PHASE 6 -- EXPLAINABLE AI (Grad-CAM / attention-rollout)")
    print("=" * 70)

    rows = []
    for name in Config.XAI_MODELS:
        if name not in feature_data or name not in fitted_probes:
            print(f"  {name:20s} SKIPPED (no cached features / no fitted primary probe)")
            continue

        in_sel, ho_sel = select_xai_samples(
            feature_data, name, fitted_probes, Config.XAI_N_SAMPLES, Config.XAI_SEED)
        if len(in_sel) == 0 and len(ho_sel) == 0:
            print(f"  {name:20s} SKIPPED (no correctly-classified tumour patches found)")
            continue

        print(f"\n  [{name}] {len(in_sel)} in-domain + {len(ho_sel)} held-out-centre samples")
        try:
            model, input_size = _load_xai_model(name, device)
        except Exception as e:
            print(f"  {name:20s} X could not reload for XAI: {e}")
            continue

        eff_w = eff_b = None
        if Config.XAI_MODELS[name]['type'] == 'cam':
            clf, scaler = fitted_probes[name]
            eff_w, eff_b = fold_scaler_into_linear(clf, scaler)

        tfm = transforms.Compose([
            transforms.Resize((input_size, input_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

        fig, axes = plt.subplots(2, Config.XAI_N_SAMPLES, figsize=(3 * Config.XAI_N_SAMPLES, 6.4))
        if axes.ndim == 1:
            axes = axes.reshape(2, -1)

        for split_i, (split_name, sel) in enumerate([('in_domain', in_sel), ('held_out_centre', ho_sel)]):
            for col in range(Config.XAI_N_SAMPLES):
                ax = axes[split_i, col]
                ax.axis('off')
                if col >= len(sel):
                    continue
                idx = int(sel[col])
                item = hf_dataset[idx]
                img = item['image']
                if img.mode != 'RGB':
                    img = img.convert('RGB')
                x = tfm(img).unsqueeze(0).to(device)

                try:
                    heat = explain_one(name, model, x, device, eff_w, eff_b)
                except Exception as e:
                    ax.set_title("XAI failed", fontsize=7, color='red')
                    print(f"    idx={idx} X explain failed: {e}")
                    continue

                ratio = center_mass_ratio(heat)
                rows.append({'model': name, 'split': split_name, 'patch_index': idx,
                            'center_mass_ratio': ratio})

                disp = np.array(img.resize((input_size, input_size)))
                heat_img = PILImage.fromarray((heat * 255).astype(np.uint8)).resize(
                    (input_size, input_size), resample=PILImage.BILINEAR)
                heat_norm = np.array(heat_img).astype(np.float32) / 255.0

                ax.imshow(disp)
                ax.imshow(heat_norm, cmap='jet', alpha=0.45)
                ax.set_title(f"cm={ratio:.2f}", fontsize=8)
            axes[split_i, 0].set_ylabel(split_name, fontsize=10)

        fig.suptitle(f"{name} -- Grad-CAM/attention overlay, in-domain (top) vs "
                      f"held-out centre {Config.TEST_CENTER} (bottom)\n"
                      f"cm = center-mass ratio (label-defining region), area fraction "
                      f"~{Config.XAI_CENTER_FRAC**2:.2f}", fontsize=9)
        plt.tight_layout()
        out_path = f"{Config.XAI_DIR}/xai_{name}_{timestamp}.png"
        plt.savefig(out_path, dpi=140)
        plt.close(fig)
        print(f"    OK {out_path}")

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    xai_df = pd.DataFrame(rows)
    if not xai_df.empty:
        csv_path = f"{Config.XAI_DIR}/center_mass_ratio_{timestamp}.csv"
        xai_df.to_csv(csv_path, index=False)
        print(f"\nOK {csv_path}")

        summary = xai_df.groupby(['model', 'split'])['center_mass_ratio'].mean().unstack('split')
        print("\nCenter-mass ratio (mean, higher = more focused on the label-defining region):")
        print(summary.to_string())

        fig, ax = plt.subplots(figsize=(10, 5))
        summary.plot(kind='bar', ax=ax)
        ax.axhline(Config.XAI_CENTER_FRAC ** 2, ls='--', c='grey',
                  label=f'region area fraction ({Config.XAI_CENTER_FRAC**2:.2f})')
        ax.set_ylabel('mean center-mass ratio')
        ax.set_title('Explanation focus on the label-defining region: in-domain vs held-out centre')
        ax.legend()
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        bar_path = f"{Config.XAI_DIR}/center_mass_summary_{timestamp}.png"
        plt.savefig(bar_path, dpi=150)
        plt.close(fig)
        print(f"OK {bar_path}")

    return xai_df


# ====================================================================
# MAIN
# ====================================================================
def main():
    """
    NOTEBOOK 2 of 3 -- Phase A (diagnostics) + Phase 2 (primary evaluation) +
    Phase 3 (multi-seed leave-one-centre-out). All frozen-feature work, no raw
    images needed, so it does NOT reload Camelyon17-WILDS -- it just reuses
    Notebook 1's cached per-model features. Should take roughly 3.5-4h based
    on the previous run's pacing for this many models/seeds/centres --
    comfortably inside the 12h limit.

    BEFORE RUNNING: attach Notebook 1's output as a Kaggle input dataset and
    point PHASE1_INPUT_DIR (right below) at wherever it mounts Notebook 1's
    /kaggle/working/features folder -- e.g. "/kaggle/input/<notebook-1-output-slug>".
    """
    set_seed(Config.SEED)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # ---- point this at Notebook 1's attached output dataset ----
    PHASE1_INPUT_DIR = "/kaggle/input/datasets/setu295452"  # <-- set to your Notebook 1 output dataset
    prior_features_dir = f"{PHASE1_INPUT_DIR}/features"

    print("=" * 70)
    print("DOWNSTREAM CLASSIFICATION -- NOTEBOOK 2/3: PHASE A + 2 + 3")
    print("(diagnostics + primary eval + multi-seed LOCO, frozen features only)")
    print("=" * 70)

    def find_pkl_source_dir(start_dir, max_depth=4):
        """
        Kaggle datasets sometimes preserve the whole folder layout you uploaded
        (e.g. if you zipped/uploaded the entire /kaggle/working tree from Notebook 1,
        the .pkl files can end up nested one level deeper than expected, as an extra
        'features/features/' -- this crashed with IsADirectoryError before, trying to
        shutil.copy2() that inner 'features' folder as if it were a file). This walks
        down from `start_dir` (at most `max_depth` levels) looking for the first
        directory that actually contains .pkl files directly, instead of assuming a
        fixed nesting depth.
        """
        current = start_dir
        for _ in range(max_depth):
            if not os.path.isdir(current):
                return None
            entries = os.listdir(current)
            if any(e.endswith('.pkl') for e in entries):
                return current
            subdirs = [e for e in entries if os.path.isdir(os.path.join(current, e))]
            if len(subdirs) == 1:
                # Single subdirectory and no .pkl here -- almost certainly the extra
                # nesting level (e.g. a repeated 'features' folder) -- descend into it.
                current = os.path.join(current, subdirs[0])
                continue
            return current  # give up descending; let the caller report what it found
        return current

    os.makedirs(Config.FEATURE_DIR, exist_ok=True)
    actual_src_dir = find_pkl_source_dir(prior_features_dir)
    if actual_src_dir and os.path.isdir(actual_src_dir):
        if actual_src_dir != prior_features_dir:
            print(f"FLAG expected .pkl files directly in {prior_features_dir}, but found "
                  f"them nested one level deeper -- using {actual_src_dir} instead.")
        import shutil
        copied, skipped = 0, []
        for fn in os.listdir(actual_src_dir):
            src_p = os.path.join(actual_src_dir, fn)
            if not fn.endswith('.pkl') or not os.path.isfile(src_p):
                skipped.append(fn)
                continue
            dst_p = os.path.join(Config.FEATURE_DIR, fn)
            if not os.path.exists(dst_p):
                shutil.copy2(src_p, dst_p)
            copied += 1
        print(f"OK copied {copied} cached feature file(s) from {actual_src_dir} -> {Config.FEATURE_DIR}")
        if skipped:
            print(f"   (skipped non-.pkl entries: {skipped})")
    else:
        print(f"FLAG no .pkl files found under {prior_features_dir} (or it doesn't exist) -- "
              f"features will be re-extracted from scratch (slow, needs the raw dataset too). "
              f"Check PHASE1_INPUT_DIR above and the actual folder layout of your attached dataset.")

    # Expected model names -- every one of these needs a .pkl already sitting in
    # Config.FEATURE_DIR for the fast, no-raw-images path below to work.
    expected_models = (list(Config.PRETRAINED_MODELS.keys())
                       + ['baseline_imagenet', 'baseline_random']
                       + list(Config.FOUNDATION_MODELS.keys()))
    missing = [m for m in expected_models
               if not os.path.exists(f"{Config.FEATURE_DIR}/{m}.pkl")]

    if not missing:
        # Every model's .pkl is already cached (just copied above) -- extract_all_
        # features() hits its own cache for all of them and returns in seconds, no
        # raw-image loading, no re-extraction, no GPU forward passes needed.
        print("OK all expected feature caches present -- loading from cache only "
              "(no dataset reload needed).")
        feature_data = extract_all_features(None, [])
    else:
        # Something's missing (e.g. Notebook 1 didn't finish, or the wrong dataset
        # was attached) -- fall back to reloading the raw dataset so extract_all_
        # features() can actually recompute whatever's absent, instead of crashing
        # on a None hf_dataset.
        print(f"FLAG missing cached features for: {missing} -- reloading raw "
              f"dataset so these can be recomputed (slower).")
        hf_dataset = load_full_dataset()
        indices = list(range(len(hf_dataset)))
        feature_data = extract_all_features(hf_dataset, indices)

    if not feature_data:
        print("\nX No cached features found -- cannot continue. Check PHASE1_INPUT_DIR.")
        return

    print(f"\nOK {len(feature_data)} models loaded from cache:")
    for name, d in feature_data.items():
        print(f"   {name:25s}  shape={d['features'].shape}")

    print("\n" + "=" * 70)
    print("PHASE A: DIAGNOSTICS")
    print("=" * 70)
    diag_random = diagnose_random_baseline(feature_data)
    diag_std = diagnose_standardization(feature_data)
    diag_ckpt = test_checkpoint_integrity()
    multi_seed = run_multi_seed(feature_data)

    print("\n" + "=" * 70)
    print("PHASE 2: PRIMARY EVALUATION (Center 2 held out)")
    print("=" * 70)
    fitted_probes = {}
    primary_results = evaluate_all(
        feature_data, Config.TRAIN_CENTERS, Config.TEST_CENTER,
        tag='primary', seed=Config.SEED, fitted_probes=fitted_probes)

    print("\n" + "=" * 70)
    print("PHASE 3: LEAVE-ONE-CENTRE-OUT  (+ MULTI-SEED, item (E))")
    print("=" * 70)
    if Config.RUN_MULTISEED_LOCO:
        multiseed_lco_df, multiseed_lco_summary = run_multiseed_loco(feature_data, seeds=Config.SEEDS)
        lco_results = (multiseed_lco_df[multiseed_lco_df['seed'] == Config.SEED].to_dict('records')
                      if not multiseed_lco_df.empty else [])
    else:
        multiseed_lco_df, multiseed_lco_summary = pd.DataFrame(), pd.DataFrame()
        lco_results = leave_one_centre_out(feature_data, seed=Config.SEED)

    print("\n" + "=" * 70)
    print("SAVING PHASE A/2/3 CHECKPOINT FOR NOTEBOOK 3")
    print("=" * 70)
    checkpoint = {
        'timestamp': timestamp,
        'fitted_probes': fitted_probes,
        'primary_results': primary_results,
        'lco_results': lco_results,
        'multiseed_lco_df': multiseed_lco_df,
        'multiseed_lco_summary': multiseed_lco_summary,
        'multi_seed': multi_seed,
        'diagnostics': {
            'random_baseline': diag_random,
            'standardization': diag_std,
            'checkpoint': diag_ckpt,
            'multi_seed': multi_seed,
        },
    }
    ckpt_path = f"{Config.RESULTS_DIR}/phase23_checkpoint.pkl"
    with open(ckpt_path, 'wb') as f:
        pickle.dump(checkpoint, f)
    print(f"OK checkpoint -> {ckpt_path}")

    print("\n" + "=" * 70)
    print("PRIMARY RESULTS (Center 2 held out)")
    print("=" * 70)
    prim_df = pd.DataFrame(primary_results)
    if not prim_df.empty:
        cols = ['model', 'auroc', 'auroc_ci_lo', 'auroc_ci_hi', 'f1', 'balanced_accuracy',
                'sensitivity', 'specificity', 'mcc', 'n_train', 'n_test']
        cols = [c for c in cols if c in prim_df.columns]
        print(prim_df[cols].sort_values('auroc', ascending=False).to_string(index=False))

    if not multiseed_lco_summary.empty:
        print("\n" + "=" * 70)
        print(f"LEAVE-ONE-CENTRE-OUT, MULTI-SEED (seeds={Config.SEEDS}) -- item (E)")
        print("=" * 70)
        print(multiseed_lco_summary.sort_values(['model', 'test_center']).to_string(index=False))

    print("\n" + "=" * 70)
    print("NOTEBOOK 2 DONE.")
    print(f"   Checkpoint : {ckpt_path}")
    print("Attach this notebook's /kaggle/working output (which also still carries")
    print("the copied features/ folder) as an input dataset to Notebook 3.")
    print("=" * 70)

    return feature_data, checkpoint


# ====================================================================
# RUN
# ====================================================================
feature_data, phase23_checkpoint = main()

DOWNSTREAM CLASSIFICATION -- NOTEBOOK 2/3: PHASE A + 2 + 3
(diagnostics + primary eval + multi-seed LOCO, frozen features only)
FLAG expected .pkl files directly in /kaggle/input/datasets/setu295452/features, but found them nested one level deeper -- using /kaggle/input/datasets/setu295452/features/features instead.
OK copied 8 cached feature file(s) from /kaggle/input/datasets/setu295452/features/features -> /kaggle/working/features
OK all expected feature caches present -- loading from cache only (no dataset reload needed).

Device: cuda

[recon_resnet18] loading cached features

[recon_vit_small] loading cached features

[recon_swin_tiny] loading cached features

[baseline_imagenet] loading cache

[baseline_random] loading cache

[kaiko_vits16] loading cached features

[phikon] loading cached features

[lunit_vits16] loading cached features

OK 8 models loaded from cache:
   recon_resnet18             shape=(455954, 512)
   recon_vit_small            shape=(455954, 384)
   recon_swi